In [ ]:
# hh_regression_with_parametricKL_and_MC.py
# Hodgkin-Huxley parameter estimation (g_L & E_L)
# - Parametric-KL via MLE (scipy .fit)
# - Monte-Carlo objective evaluation (RMSE + parametric KL)
# - Option to use instructor time.xlsx file for interpolation
# - Saves results and plots
# Run: python hh_regression_with_parametricKL_and_MC.py  (or execute cells in Colab)
# Ensure scikit-learn is available in this notebook environment

from sklearn.linear_model import LinearRegression # type: ignore
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import pandas as pd
from scipy.stats import entropy, norm, gamma, lognorm, expon, laplace
from scipy.interpolate import interp1d
import os
import warnings
warnings.filterwarnings("ignore")

# ----------------------
# USER CONFIG (edit these)
# ----------------------
FILE_PATH = "file path"   # path to experimental data (Excel/CSV)
COLUMN_NUMBER = 3
DT = 0.01                # time-step for data & estimation (s)
WINDOW_SIZE = 0.1        # window size for local estimation (s)
STEP_SIZE = 0.01         # step between windows (s) -> matches required 0.01s interval
TOTAL_TIME = 5000.0      # total time for synthetic MC dataset (s)
USE_SYNTHETIC_TEST = False   # If True, runs a synthetic Monte-Carlo test instead of reading file
POISSON_RATE = 5.0      # spikes per second (adjustable). Poisson spike process rate.
SPIKE_MAG = 2.0         # µA/cm^2 spike magnitude
SPIKE_WIDTH = 0.001     # spike width in seconds (1 ms)

# Monte-Carlo sampling config for objective evaluation (replace grid search)
MC_N_SAMPLES = 500      # number of Monte-Carlo samples (increase for final runs)
MC_SEED = 2025
MC_gL_bounds = (0.01, 1.5)   # bounds for g_L sampling
MC_EL_bounds = (E_L_default - 20 if 'E_L_default' in globals() else -69, E_L_default + 20 if 'E_L_default' in globals() else -29)  # overwritten below

# Parametric candidate distributions for MLE (choose at least 'normal' and 'gamma' as per instructor)
CANDIDATE_DISTS = {
    'normal': norm,
    'gamma': gamma,
    'lognormal': lognorm,
    'exponential': expon,
    'laplace': laplace
}

# Path to instructor's time.xlsx (if provided). If None, code uses generated uniform times.
TIME_FILE_PATH = "/content/drive/MyDrive/time.xlsx"   # e.g. "/content/drive/MyDrive/teacher_time.xlsx" or None

OUTPUT_DIR = "./hh_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ----------------------

# ----------------------
# Constants (model params)
# ----------------------
Cm = 1.0
g_Na = 120.0
g_K = 36.0
g_L_default = 0.3
E_Na = 55.0
E_K = -72.0
E_L_default = -49.0

# Ensure MC_EL_bounds defaults are set now that E_L_default is defined
MC_EL_bounds = (E_L_default - 20, E_L_default + 20)

# gating functions (standard Hodgkin-Huxley-ish forms)
def alpha_n(V): return 0.01*(V+55)/(1 - np.exp(-(V+55)/10) + 1e-12)
def beta_n(V): return 0.125*np.exp(-(V+65)/80)
def alpha_m(V): return 0.1*(V+40)/(1 - np.exp(-(V+40)/10) + 1e-12)
def beta_m(V): return 4*np.exp(-(V+65)/18)
def alpha_h(V): return 0.07*np.exp(-(V+65)/20)
def beta_h(V): return 1 / (1 + np.exp(-(V+35)/10))

def I_Na(V, m, h, gNa=g_Na, ENa=E_Na): return gNa * (m**3) * h * (V - ENa)
def I_K(V, n, gK=g_K, EK=E_K): return gK * (n**4) * (V - EK)
def I_L(V, gL, EL): return gL * (V - EL)

# full ODE for simulation (V, m, h, n)
def dALLdt(X, t, I_inj_func, gL, EL):
    V, m, h, n = X
    I_inj = I_inj_func(t)
    dVdt = (I_inj - I_Na(V, m, h) - I_K(V, n) - I_L(V, gL, EL)) / Cm
    dmdt = alpha_m(V)*(1 - m) - beta_m(V)*m
    dhdt = alpha_h(V)*(1 - h) - beta_h(V)*h
    dndt = alpha_n(V)*(1 - n) - beta_n(V)*n
    return [dVdt, dmdt, dhdt, dndt]

# helper to evolve gating variables given a V trace (used for regression feature calc)
def solve_gating_vars(V_trace, t_trace):
    def dGatingsdt(X, t):
        m, h, n = X
        V = np.interp(t, t_trace, V_trace)
        dmdt = alpha_m(V)*(1 - m) - beta_m(V)*m
        dhdt = alpha_h(V)*(1 - h) - beta_h(V)*h
        dndt = alpha_n(V)*(1 - n) - beta_n(V)*n
        return [dmdt, dhdt, dndt]
    X0 = [0.05, 0.6, 0.32]
    X = odeint(dGatingsdt, X0, t_trace)
    return X[:,0], X[:,1], X[:,2]

# Generate Poisson spikes sampled at dt: returns I_ext array
def generate_poisson_spikes(n_points, dt, rate=POISSON_RATE, spike_mag=SPIKE_MAG, spike_width=SPIKE_WIDTH):
    p = rate * dt
    events = np.random.rand(n_points) < p
    I_ext = np.zeros(n_points)
    spike_samples = max(1, int(np.round(spike_width / dt)))
    for i, ev in enumerate(events):
        if ev:
            end = min(n_points, i + spike_samples)
            I_ext[i:end] += spike_mag
    return I_ext

# Monte-Carlo time-varying parameter generator (random walk + periodic)
def monte_carlo_time_varying_params(t, base_gL=g_L_default, base_EL=E_L_default, seed=None):
    rng = np.random.RandomState(seed)
    n = len(t)
    gL = np.ones(n) * base_gL
    EL = np.ones(n) * base_EL
    step_std_g = 0.03 * base_gL
    step_std_EL = 0.5
    for i in range(1, n):
        gL[i] = gL[i-1] + rng.normal(scale=step_std_g)
        EL[i] = EL[i-1] + rng.normal(scale=step_std_EL)
    gL += 0.05 * base_gL * np.sin(2*np.pi*0.001*t)
    EL += 1.0 * np.sin(2*np.pi*0.002*t)
    gL = np.clip(gL, 0.01, 5.0)
    EL = np.clip(EL, -100.0, 20.0)
    return gL, EL

# Windowed estimation for g_L and E_L using linear regression
def estimate_gL_EL_windowed(t_sec, V, I_func, window_size_sec, step_size_sec):
    estimated_times = []
    estimated_gL = []
    estimated_EL = []

    if len(t_sec) < 2:
        return [], [], []

    dt = t_sec[1] - t_sec[0]
    if dt <= 0:
        return [], [], []

    dVdt = np.gradient(V, dt)
    m, h, n = solve_gating_vars(V, t_sec)
    I_inj_trace = I_func(t_sec)
    I_Na_trace = I_Na(V, m, h)
    I_K_trace = I_K(V, n)
    Y_target = (Cm * dVdt) - I_inj_trace + I_Na_trace + I_K_trace  # should equal I_L

    X_features = np.vstack([V, np.ones_like(V)]).T

    n_points = len(t_sec)
    window_pts = int(round(window_size_sec / dt))
    step_pts = int(round(step_size_sec / dt))
    if window_pts <= 0: window_pts = 1
    if step_pts <= 0: step_pts = 1
    if window_pts > n_points:
        window_pts = n_points

    model = LinearRegression(fit_intercept=False)

    for i in range(0, n_points - window_pts + 1, step_pts):
        win_start = i
        win_end = i + window_pts
        t_window_mid = t_sec[win_start + window_pts // 2]
        Y_win = Y_target[win_start:win_end]
        X_win = X_features[win_start:win_end, :]
        if np.allclose(X_win[:,0], X_win[0,0]):
            continue
        try:
            model.fit(X_win, Y_win)
            coef_V = model.coef_[0]
            intercept = model.coef_[1]
            if abs(coef_V) < 1e-8:
                continue
            gL_est = coef_V
            EL_est = -intercept / gL_est
            if 0.0 < gL_est < 50.0 and -200.0 < EL_est < 100.0:
                estimated_times.append(t_window_mid)
                estimated_gL.append(gL_est)
                estimated_EL.append(EL_est)
        except Exception:
            pass

    return estimated_times, estimated_gL, estimated_EL

# Simulation driver for a single (constant) parameter set over the time vector t_short
def run_simulation_const_params(params, t_short, I_inj_short, X0):
    gL, EL = params
    try:
        X = odeint(dALLdt, X0, t_short, args=(I_inj_short, gL, EL))
        V_sim = X[:,0]
        if not np.all(np.isfinite(V_sim)):
            return None
        return V_sim
    except Exception:
        return None

# Objective RMSE
def objective_rmse(V_short, V_sim):
    if V_sim is None:
        return 1e9
    return np.sqrt(np.mean((V_short - V_sim)**2))

# -------------------------
# Parametric distribution fitting + parametric KL
# -------------------------
def fit_best_parametric_distribution(data, candidates=CANDIDATE_DISTS):
    best_name = None
    best_dist = None
    best_params = None
    best_ll = -np.inf
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    if data.size == 0:
        return None, None, None, -np.inf
    for name, dist in candidates.items():
        try:
            # handle distributions with domain restrictions: gamma requires positive data
            if name == 'gamma':
                # shift if data contains negatives: fit gamma to shifted data
                if np.any(data <= 0):
                    shift = np.min(data) - 1e-3
                    params = dist.fit(data - shift)
                    # store shift as part of params (we'll return (shift, params) for gamma)
                    params = (shift,) + params
                else:
                    params = dist.fit(data)
            else:
                params = dist.fit(data)
            # compute log-likelihood robustly
            try:
                # For gamma with shift included, compute logpdf carefully
                if name == 'gamma' and params is not None and isinstance(params, tuple) and len(params) > 0 and isinstance(params[0], (float, np.floating)):
                    # We used shift as first element
                    shift = params[0]
                    gamma_params = params[1:]
                    ll = np.sum(dist.logpdf(data - shift, *gamma_params))
                else:
                    ll = np.sum(dist.logpdf(data, *params))
            except Exception:
                ll = -np.inf
            if ll > best_ll:
                best_ll = ll
                best_name = name
                best_dist = dist
                best_params = params
        except Exception:
            continue
    return best_name, best_dist, best_params, best_ll

def parametric_pdf(dist, params, x):
    # if gamma stored with shift as first param
    if dist == gamma and isinstance(params, tuple) and len(params) > 0 and params[0] is not None and (params[0] < -1e-6 or params[0] > -1e-6):
        shift = params[0]
        gamma_params = params[1:]
        return dist.pdf(x - shift, *gamma_params)
    else:
        return dist.pdf(x, *params)

def kl_between_parametric(distP, paramsP, distQ, paramsQ, x_min=None, x_max=None, npoints=2000, eps=1e-12):
    # pick integration bounds from P
    try:
        # attempt to compute mean/var if available
        mu = distP.mean(*paramsP) if hasattr(distP, 'mean') else np.mean
        sd = np.sqrt(distP.var(*paramsP)) if hasattr(distP, 'var') else None
        # fallback numeric bounds
        if sd is None or not np.isfinite(sd):
            x_min, x_max = -120, 80
        else:
            x_min = mu - 6*sd
            x_max = mu + 6*sd
    except Exception:
        x_min, x_max = -120, 80
    if x_min is None or x_max is None:
        x_min, x_max = -120, 80
    x = np.linspace(x_min, x_max, npoints)
    p = parametric_pdf(distP, paramsP, x) + eps
    q = parametric_pdf(distQ, paramsQ, x) + eps
    # normalize numerically
    p = p / (np.trapz(p, x))
    q = q / (np.trapz(q, x))
    integrand = p * np.log(p / q)
    kl = np.trapz(integrand, x)
    return float(kl)

# Helper: attempt to load instructor time file
def load_teacher_time(time_file_path):
    if time_file_path is None:
        return None
    try:
        df = pd.read_excel(time_file_path, header=None)
        # assume first column is time
        time = df.iloc[:,0].astype(float).values
        # ensure strictly increasing
        if np.any(np.diff(time) <= 0):
            time = np.sort(time)
        return time
    except Exception as e:
        print("Could not load teacher time file:", e)
        return None

# Monte-Carlo sampling of objective (parametric-KL)
def monte_carlo_objective_sampling(t_short, V_short, I_inj_short, X0,
                                   gL_bounds=MC_gL_bounds, EL_bounds=MC_EL_bounds,
                                   n_samples=MC_N_SAMPLES, rng_seed=MC_SEED, fit_parametric_for_KL=True):
    rng = np.random.RandomState(rng_seed)
    samples = rng.rand(n_samples, 2)
    gL_samples = gL_bounds[0] + samples[:,0] * (gL_bounds[1] - gL_bounds[0])
    EL_samples = EL_bounds[0] + samples[:,1] * (EL_bounds[1] - EL_bounds[0])
    results = []
    # fit experimental parametric once
    if fit_parametric_for_KL:
        best_name_exp, best_dist_exp, best_params_exp, ll_exp = fit_best_parametric_distribution(V_short)
    else:
        best_dist_exp = None
        best_params_exp = None
    for gL, EL in zip(gL_samples, EL_samples):
        V_sim = run_simulation_const_params((gL, EL), t_short, I_inj_short, X0)
        rmse = objective_rmse(V_short, V_sim)
        if best_dist_exp is None or V_sim is None:
            kl_val = 1e9
        else:
            best_name_sim, best_dist_sim, best_params_sim, ll_sim = fit_best_parametric_distribution(V_sim)
            if best_dist_sim is None:
                kl_val = 1e9
            else:
                kl_val = kl_between_parametric(best_dist_exp, best_params_exp, best_dist_sim, best_params_sim)
        results.append((gL, EL, rmse, kl_val))
    df = pd.DataFrame(results, columns=['gL', 'EL', 'rmse', 'kl'])
    return df

# Short mathematical note about KL continuity (printed)
def print_kl_note():
    print("\nKL-Divergence note:")
    print("KL(P || Q) = ∫ p(x) log( p(x) / q(x) ) dx")
    print("We fit parametric PDFs to exp & sim via MLE; numerically integrate to compute KL.")
    print("Small epsilon added to pdf values to avoid log(0); integration bounds set from fitted P.\n")

# Main analysis
def run_analysis(file_path, column_number, dt, window_size, step_size, use_synthetic=False, teacher_time_path=None):
    print("="*70)
    print("HODGKIN-HUXLEY PARAMETER ESTIMATION (g_L, E_L) - WITH PARAMETRIC KL & MONTE-CARLO")
    print("="*70)
    print(f"Time-step: {dt} s | Window: {window_size} s | Step: {step_size} s\n")

    teacher_time = load_teacher_time(teacher_time_path)
    if teacher_time is not None:
        print("Using teacher-provided time vector from:", teacher_time_path)
    else:
        print("No teacher time file provided; using uniform time grid where needed.")

    if use_synthetic:
        print("Generating synthetic Monte-Carlo dataset (time-varying g_L and E_L)...")
        if teacher_time is not None:
            t = teacher_time
        else:
            t = np.arange(0, TOTAL_TIME, dt)
        n_pts = len(t)
        gL_series, EL_series = monte_carlo_time_varying_params(t, seed=42)
        I_ext = generate_poisson_spikes(n_pts, dt, rate=POISSON_RATE)
        X = np.zeros((n_pts, 4))
        X[0,:] = [-65.0, 0.0529, 0.5961, 0.3177]
        for i in range(n_pts - 1):
            t_span = [t[i], t[i+1]]
            sol = odeint(dALLdt, X[i,:], t_span, args=(lambda tt: float(np.interp(tt, t, I_ext)), gL_series[i], EL_series[i]))
            X[i+1,:] = sol[-1,:]
        V_exp = X[:,0]
        t_exp_sec = t
        print("Synthetic dataset generated.")
    else:
        try:
            if file_path.endswith('.csv'):
                data = pd.read_csv(file_path, header=None)
            else:
                data = pd.read_excel(file_path, header=None, skiprows=3)
            col_idx = 2 * int(column_number) - 2
            if col_idx >= len(data.columns):
                print(f"ERROR: Column {column_number} does not exist in the data.")
                return
            V_exp = data.iloc[:, col_idx].astype(float).values
            n_points = len(V_exp)
            if teacher_time is not None:
                # If teacher_time length differs, interpolate V_exp to teacher_time (or vice-versa)
                if len(teacher_time) == n_points:
                    t_exp_sec = teacher_time
                else:
                    # create uniform time vector matching dt and interpolate teacher_time to it if needed
                    t_exp_sec = np.arange(0, n_points * dt, dt)
                    # if teacher_time is intended as interpolation base, resample V_exp to teacher_time:
                    try:
                        interpV = interp1d(t_exp_sec, V_exp, bounds_error=False, fill_value="extrapolate")
                        V_exp = interpV(teacher_time)
                        t_exp_sec = teacher_time
                    except Exception:
                        # fallback
                        t_exp_sec = np.arange(0, n_points * dt, dt)
                # note: instructor guidance may require different interpolation; check teacher note
            else:
                t_exp_sec = np.arange(0, n_points * dt, dt)
            I_ext = generate_poisson_spikes(len(t_exp_sec), dt, rate=POISSON_RATE)
            print(f"Loaded file {file_path} ({n_points} points). Using time vector length {len(t_exp_sec)}.")
        except Exception as e:
            print("Error loading file:", e)
            return

    I_inj_func = interp1d(t_exp_sec, I_ext, fill_value="extrapolate")

    # 1) Dynamic parameter estimation (sliding-window)
    print("\n[1/3] Running dynamic (windowed) parameter estimation for g_L and E_L...")
    est_t, est_gL, est_EL = estimate_gL_EL_windowed(t_exp_sec, V_exp, I_inj_func, window_size, step_size)

    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(10,6), sharex=True)
    if est_t:
        ax1.plot(est_t, est_gL, '.-', label='Estimated g_L')
        ax2.plot(est_t, est_EL, '.-', label='Estimated E_L')
        print(f"Found {len(est_t)} estimates.")
    else:
        print("No valid estimates found in the window setup — try increasing window size.")
    ax1.set_ylabel('g_L (mS/cm^2)')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('E_L (mV)')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()
    fn_est = os.path.join(OUTPUT_DIR, 'estimates_gL_EL.png')
    plt.savefig(fn_est, dpi=300, bbox_inches='tight')
    plt.show()
    # save dynamic estimates csv
    if est_t:
        df_est = pd.DataFrame({'time': est_t, 'gL': est_gL, 'EL': est_EL})
        df_est.to_csv(os.path.join(OUTPUT_DIR, 'dynamic_estimates_gL_EL.csv'), index=False)
        print("Saved dynamic estimates CSV.")

    # 2) Distribution analysis (fit parametric distribution to experimental data)
    print("\n[2/3] Analyzing voltage distribution (experiment) and performing parametric MLE fits...")
    V_no_outliers = V_exp[np.abs(V_exp - np.mean(V_exp)) < 3 * np.std(V_exp)]
    if V_no_outliers.size > 0:
        x_range = np.linspace(np.min(V_no_outliers), np.max(V_no_outliers), 500)
        fig2, axes = plt.subplots(1,3, figsize=(15,4))
        axes[0].hist(V_no_outliers, bins=100, density=True, histtype='stepfilled', alpha=0.7)
        # KDE for visualization only (not used in KL)
        try:
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(V_no_outliers)
            axes[1].plot(x_range, kde(x_range))
        except Exception:
            axes[1].text(0.5,0.5,"KDE unavailable", ha='center')
        mu, std = norm.fit(V_no_outliers)
        axes[2].plot(x_range, norm.pdf(x_range, mu, std), '--')
        axes[0].set_title('Histogram (visual)')
        axes[1].set_title('KDE (visual)')
        axes[2].set_title('Gaussian fit (visual)')
        plt.tight_layout()
        fn_vd = os.path.join(OUTPUT_DIR, 'voltage_dists.png')
        plt.savefig(fn_vd, dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("No valid voltage data for distribution analysis.")

    # Fit best parametric distribution for the experimental short window used for MC (below)
    # We'll use the first t_analysis seconds (short slice) for MC objective to keep runtime manageable
    print_kl_note()

    # 3) Monte-Carlo objective (parametric-KL + RMSE)
    print("\n[3/3] Monte-Carlo sampling of objective (RMSE & parametric-KL)...")
    # choose short analysis window for objective computation (1s or full if short)
    t_analysis_end = min(1.0, t_exp_sec[-1]) if len(t_exp_sec) > 1 else t_exp_sec[-1]
    idx_end = np.searchsorted(t_exp_sec, t_analysis_end)
    if idx_end <= 1:
        idx_end = len(t_exp_sec)
    t_short = t_exp_sec[:idx_end]
    V_short = V_exp[:idx_end]
    I_inj_short = interp1d(t_short, I_ext[:idx_end], fill_value="extrapolate")
    X0 = [-65.0, 0.05, 0.6, 0.32]

    # Monte-Carlo sampling (parametric KL computed inside)
    df_mc = monte_carlo_objective_sampling(t_short, V_short, I_inj_short, X0,
                                           gL_bounds=MC_gL_bounds, EL_bounds=MC_EL_bounds,
                                           n_samples=MC_N_SAMPLES, rng_seed=MC_SEED, fit_parametric_for_KL=True)
    fn_mc = os.path.join(OUTPUT_DIR, 'monte_carlo_objective_results.csv')
    df_mc.to_csv(fn_mc, index=False)
    print(f"Monte-Carlo results saved to {fn_mc} (n={len(df_mc)})")

    # Visualize Monte-Carlo results: scatter colored by RMSE and KL
    plt.figure(figsize=(6,5))
    sc = plt.scatter(df_mc['EL'], df_mc['gL'], c=df_mc['rmse'], cmap='viridis', s=28, edgecolor='k', linewidth=0.2)
    plt.colorbar(sc, label='RMSE')
    plt.xlabel('E_L (mV)'); plt.ylabel('g_L (mS/cm^2)')
    plt.title('Monte-Carlo RMSE samples')
    fn_mc_rmse = os.path.join(OUTPUT_DIR, 'montecarlo_rmse_scatter.png')
    plt.savefig(fn_mc_rmse, dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(6,5))
    sc2 = plt.scatter(df_mc['EL'], df_mc['gL'], c=df_mc['kl'], cmap='magma', s=28, edgecolor='k', linewidth=0.2)
    plt.colorbar(sc2, label='KL (parametric)')
    plt.xlabel('E_L (mV)'); plt.ylabel('g_L (mS/cm^2)')
    plt.title('Monte-Carlo KL samples')
    fn_mc_kl = os.path.join(OUTPUT_DIR, 'montecarlo_kl_scatter.png')
    plt.savefig(fn_mc_kl, dpi=300, bbox_inches='tight')
    plt.show()

    print("\nAnalysis complete. Saved outputs in:", OUTPUT_DIR)
    print(" -", fn_est)
    print(" -", fn_vd if 'fn_vd' in locals() else "(no voltage dist plot)")
    print(" -", fn_mc)
    print(" -", fn_mc_rmse)
    print(" -", fn_mc_kl)
    print("="*70)

# run main if script executed
if __name__ == "__main__":
    run_analysis(FILE_PATH, COLUMN_NUMBER, DT, WINDOW_SIZE, STEP_SIZE, use_synthetic=USE_SYNTHETIC_TEST, teacher_time_path=TIME_FILE_PATH)


HODGKIN-HUXLEY PARAMETER ESTIMATION (g_L, E_L) - WITH PARAMETRIC KL & MONTE-CARLO
Time-step: 0.01 s | Window: 0.1 s | Step: 0.01 s

Could not load teacher time file: [Errno 2] No such file or directory: '/content/drive/MyDrive/time.xlsx'
No teacher time file provided; using uniform time grid where needed.
Error loading file: [Errno 2] No such file or directory: 'file path'
